# NOTES

Since "Reference - Global Baseline Model - Linear Regression & Support Vector Machine" **failed**, based on the clustering + PCA as a feature selection steps, the clusters ("LEADERS" & "FOLLOWERS") were was trained on the following models -

**1. Baseline Model:** a. Linear Regression, b. Support Vector Regressor
    
**2. Timeseries Model:** a. ARIMAX (ARIMA + exogenuous variables)

In [1]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from statsmodels.tsa.stattools import arma_order_select_ic
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset (1).csv


In [3]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),BEV (New Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),...,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_BEV (New Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_Available,Log_DC Recharging Speed (km/h),YJ_Purchase price (EUR)
0,Austria,2011,0.18,631.0,308167.0,3.286579,1.6,963,17.500000,175.000000,...,95.000000,30469.93,0.165514,6.448889,6.871091,12.638400,1.455489,1.609438,5.170484,-1.493084
1,Austria,2012,0.13,427.0,316589.4,2.485676,1.0,1063,29.169521,217.394139,...,95.000000,30469.93,0.122218,6.059123,6.969791,12.665364,1.248662,1.609438,5.386301,-1.493084
2,Austria,2013,0.21,656.0,321191.7,2.000156,0.4,1173,48.200000,270.000000,...,142.220000,30469.93,0.190620,6.487684,7.068172,12.679797,1.098664,2.197225,5.602119,-1.493084
3,Austria,2014,0.44,1344.0,330113.5,1.605812,1.0,1393,22.600000,240.000000,...,155.770000,35555.64,0.364643,7.204149,7.239933,12.707195,0.957744,2.639057,5.484797,-0.885657
4,Austria,2015,0.55,1699.0,342083.5,0.896563,0.6,1625,47.700000,381.250000,...,215.220000,42052.00,0.438255,7.438384,7.393878,12.742813,0.640043,3.091042,5.946075,0.277809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Sweden,2020,9.55,27897.0,5012855.0,0.497367,-1.3,47680,39.325301,403.164557,...,286.200000,38311.38,2.356126,10.236310,10.772288,15.427516,0.403708,4.584967,6.001822,-0.451036
374,Sweden,2021,19.09,57454.0,5417760.0,2.163197,1.3,70537,40.904348,493.391304,...,308.730000,40745.01,3.000222,10.958757,11.163907,15.505193,1.151583,5.129899,6.203327,0.003737
375,Sweden,2022,32.84,94603.0,5816415.0,8.369291,3.5,83510,49.314815,594.259259,...,345.440000,43874.27,3.521644,11.457455,11.332734,15.576195,2.237437,5.537334,6.388997,0.697159
376,Sweden,2023,38.42,111338.0,6143187.0,8.548625,1.2,126758,70.307692,649.230769,...,361.794521,45999.05,3.674273,11.620335,11.750043,15.630854,2.256397,5.918894,6.477327,1.244323


In [4]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [5]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

In [6]:
# Data splitting into Pre and Post COVID
pre_covid_df = dataset[dataset['Year'] < 2020].copy()
post_covid_df = dataset[dataset['Year'] >= 2020].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [7]:
# Check which columns have NaNs and how many
print("Missing Values Per Column (Pre-COVID)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column (Post-COVID)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column (Pre-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           8
dtype: int64

Missing Values Per Column (Post-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64


In [8]:
# Checking which rows are empty
print("Missing Log_CPI in Pre-COVID")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])
print("\n")
print("Missing Log_CPI in Post-COVID")
print(post_covid_df[post_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Pre-COVID
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7   Romania  2016      NaN


Missing Log_CPI in Post-COVID
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [9]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of Pre-COVID data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [10]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [11]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [12]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (243, 3)
Test Shape (PCs): (135, 3)


In [13]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France',
    'Germany', 'Hungary', 'Italy', 'Netherlands', 'Poland',
    'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland',
    'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [14]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Baseline Models

**1. Linear Regression**

In [15]:
def run_lr_model(X_train, y_train, X_test, y_test, cluster_name):
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    train_pred = lr.predict(X_train)
    test_pred = lr.predict(X_test)

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)

    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)

    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - Linear Regression")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'lr_model': lr,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [16]:
# Run for both clusters
results_lr_leaders = run_lr_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_lr_followers = run_lr_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - Linear Regression
Train R² (Log): 0.4512, R² (Real): 0.1899, MAE (Real): 0.46%
Test  R² (Log): -1.8812, R² (Real): -0.8344, MAE (Real): 10.56%

Followers - Linear Regression
Train R² (Log): 0.2445, R² (Real): 0.1586, MAE (Real): 0.30%
Test  R² (Log): -2.3250, R² (Real): -0.6755, MAE (Real): 5.92%



In [17]:
# Combine predictions and actual values for leaders and followers
combined_preds = np.concatenate([
    results_lr_leaders['pred_test_real'],
    results_lr_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_lr_leaders['y_test_real'],
    results_lr_followers['y_test_real']
])

# Combine metadata in the same order
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_lr_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error
all_lr_results['Error_Abs'] = (
    all_lr_results['Actual_BEV_%'] - all_lr_results['Predicted_BEV_%']
).abs()

# Print results by year and cluster
for year in sorted(all_lr_results['Year'].unique()):
    print(f"\n---> LINEAR REGRESSION: {int(year)} PREDICTION RESULTS")

    year_data = all_lr_results[all_lr_results['Year'] == year]

    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


---> LINEAR REGRESSION: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         2.139162  19.290838
        Sweden         10.55         2.021364   8.528636
       Denmark          8.16         1.826884   6.333116
        France          7.72         1.274595   6.445405
       Germany          7.56         2.055572   5.504428
       Austria          7.41         1.838461   5.571539
       Finland          5.40         1.579205   3.820795
       Belgium          4.48         1.856529   2.623471
         Italy          3.35         1.748598   1.601402
       Hungary          3.32         2.220454   1.099546
       Romania          3.25         1.743165   1.506835
         Spain          3.14         1.517574   1.622426
Czech Republic          2.61         1.985110   0.624890
        Poland          1.86         2.037949   0.177949

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  

In [18]:
# Save the results in an excel file

# Filter the results as per the clusters
all_lr_results['Cluster'] = all_lr_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_lr_results[all_lr_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_lr_results[all_lr_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial1_LR_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_LR_Followers_Results.xlsx", index=False)

**2. Support Vector Machine**

In [19]:
def run_svr_model(X_train, y_train, X_test, y_test, cluster_name):
    svr = SVR(kernel='rbf', C=100, epsilon=0.1)
    svr.fit(X_train, y_train)

    train_pred = svr.predict(X_train)
    test_pred = svr.predict(X_test)

    # Back-transform
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred)
    test_pred_real = np.exp(test_pred)

    # Compute metrics
    r2_train_log = r2_score(y_train, train_pred)
    r2_test_log = r2_score(y_test, test_pred)

    r2_train_real = r2_score(y_train_real, train_pred_real)
    r2_test_real = r2_score(y_test_real, test_pred_real)

    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - Support Vector Regression")
    print(f"Train R² (Log): {r2_train_log:.4f}, R² (Real): {r2_train_real:.4f}, MAE (Real): {mae_train_real:.2f}%")
    print(f"Test  R² (Log): {r2_test_log:.4f}, R² (Real): {r2_test_real:.4f}, MAE (Real): {mae_test_real:.2f}%\n")

    return {
        'svr_model': svr,
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [20]:
# Run for both clusters
results_svr_leaders = run_svr_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_svr_followers = run_svr_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

Leaders - Support Vector Regression
Train R² (Log): 0.7465, R² (Real): 0.8841, MAE (Real): 0.27%
Test  R² (Log): -6.7132, R² (Real): -725.2743, MAE (Real): 150.71%

Followers - Support Vector Regression
Train R² (Log): 0.7371, R² (Real): 0.7946, MAE (Real): 0.17%
Test  R² (Log): -2.3644, R² (Real): -14.0518, MAE (Real): 17.14%



In [21]:
# Combine predictions and actual values for leaders and followers
combined_preds = np.concatenate([
    results_svr_leaders['pred_test_real'],
    results_svr_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_svr_leaders['y_test_real'],
    results_svr_followers['y_test_real']
])

# Combine metadata in the same order
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_svr_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error
all_svr_results['Error_Abs'] = (
    all_svr_results['Actual_BEV_%'] - all_svr_results['Predicted_BEV_%']
).abs()

# Print results by year and cluster
for year in sorted(all_svr_results['Year'].unique()):
    print(f"\n---> SUPPORT VECTOR REGRESSION: {int(year)} PREDICTION RESULTS")

    year_data = all_svr_results[all_svr_results['Year'] == year]

    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )

    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


---> SUPPORT VECTOR REGRESSION: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         1.666529  19.763471
        Sweden         10.55         1.998913   8.551087
       Denmark          8.16         1.743023   6.416977
        France          7.72         1.438345   6.281655
       Germany          7.56         1.829377   5.730623
       Austria          7.41         1.513740   5.896260
       Finland          5.40         1.263450   4.136550
       Belgium          4.48         1.564791   2.915209
         Italy          3.35         2.706659   0.643341
       Hungary          3.32         1.570654   1.749346
       Romania          3.25         1.474739   1.775261
         Spain          3.14         6.171510   3.031510
Czech Republic          2.61         1.824542   0.785458
        Poland          1.86         2.042977   0.182977

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual

In [22]:
# Save the results in an excel file

# Filter the results as per the clusters
all_svr_results['Cluster'] = all_svr_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_svr_results[all_svr_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_svr_results[all_svr_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial1_SVR_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_SVR_Followers_Results.xlsx", index=False)

## Time-Series Models

**3. ARIMAX (with exogenous variables)**

In [23]:
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', UserWarning)

def run_arimax_model(X_train, y_train, X_test, y_test, country_name):
    # Auto-select AR and MA order based on AIC
    try:
        order_res = arma_order_select_ic(y_train, ic='aic', max_ar=2, max_ma=2)
        p, q = order_res.aic_min_order
    except Exception:
        p, q = 1, 0  # fallback
    d = 1  # integration order

    try:
        model = ARIMA(y_train, exog=X_train, order=(p, d, q)).fit()
    except Exception as e:
        print(f"Skipping {country_name}: model fitting failed ({e})")
        return None

    # Train predictions
    train_pred_log = model.predict(start=0, end=len(y_train) - 1, exog=X_train)
    train_pred_real = np.exp(train_pred_log)
    y_train_real = np.exp(y_train)

    # Test predictions
    test_pred_log = model.forecast(steps=len(y_test), exog=X_test)
    test_pred_real = np.exp(test_pred_log)
    y_test_real = np.exp(y_test)

    # Evaluate metrics on train set
    r2_train_log = r2_score(y_train, train_pred_log)
    r2_train_real = r2_score(y_train_real, train_pred_real)
    mae_train_real = mean_absolute_error(y_train_real, train_pred_real)

    # Evaluate metrics on test set
    r2_test_log = r2_score(y_test, test_pred_log)
    r2_test_real = r2_score(y_test_real, test_pred_real)
    mae_test_real = mean_absolute_error(y_test_real, test_pred_real)

    # Create Train set dataframe
    df_train = pd.DataFrame({
        'Country': country_name,
        'Year': pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_train_real.values,
        'Predicted_BEV_%': train_pred_real.values,
        'Type': 'Train',
        'Model_Order': f"({p},{d},{q})"
    })

    # Create Test set dataframe
    df_test = pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real.values,
        'Predicted_BEV_%': test_pred_real.values,
        'Type': 'Test',
        'Model_Order': f"({p},{d},{q})"
    })

    return pd.concat([df_train, df_test], ignore_index=True)


In [24]:
# Combining leaders and followers datasets
all_arimax_results_list = []
all_countries = leaders_list + followers_list

for country in all_countries:

    # Data for the specific country to train and test datframe
    # X_tr = train using features from leaders + followers, y_tr = target of leaders + followers
    # X_te = test on using features from leaders + followers, y_te = target of leaders + followers
    X_tr = X_train_pca[pre_covid_fixed['Country'] == country]
    y_tr = y_train[pre_covid_fixed['Country'] == country]
    X_te = X_test_pca[post_covid_fixed['Country'] == country]
    y_te = y_test[post_covid_fixed['Country'] == country]

    # Run both models
    res = run_arimax_model(X_tr, y_tr, X_te, y_te, country)
    all_arimax_results_list.append(res)

# Combine all individual country results
all_arimax_results = pd.concat(all_arimax_results_list, ignore_index=True)

# Calculate Absolute Error
all_arimax_results['Error_Abs'] = (
    all_arimax_results['Actual_BEV_%'] - all_arimax_results['Predicted_BEV_%']
).abs()

# Assign Clusters
all_arimax_results['Cluster'] = all_arimax_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Print results by year and cluster
for year in sorted(all_arimax_results[all_arimax_results['Type'] == 'Test']['Year'].unique()):
    print(f"\n---> ARIMAX MODEL: {int(year)} PREDICTION RESULTS")

    year_data = all_arimax_results[(all_arimax_results['Year'] == year) & (all_arimax_results['Type'] == 'Test')]

    # Leaders Cluster
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    leaders_data = year_data[year_data['Cluster'] == 'Leaders'].sort_values('Actual_BEV_%', ascending=False)
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Cluster
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    followers_data = year_data[year_data['Cluster'] == 'Followers'].sort_values('Actual_BEV_%', ascending=False)
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

# Save only test results to excel
test_results = all_arimax_results[all_arimax_results['Type'] == 'Test']

# Leaders cluster
leaders_df = test_results[test_results['Cluster'] == 'Leaders'] \
    .sort_values(['Country', 'Year'])

# Followers cluster
followers_df = test_results[test_results['Cluster'] == 'Followers'] \
    .sort_values(['Country', 'Year'])

leaders_df.to_excel("Trial1_ARIMAX_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial1_ARIMAX_Followers_Results.xlsx", index=False)


---> ARIMAX MODEL: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43        33.316247  11.886247
        Sweden         10.55        12.537650   1.987650
       Denmark          8.16         1.673257   6.486743
        France          7.72         2.966311   4.753689
       Germany          7.56         5.070690   2.489310
       Austria          7.41         6.481710   0.928290
       Finland          5.40         5.050165   0.349835
       Belgium          4.48         2.720263   1.759737
         Italy          3.35         1.790249   1.559751
       Hungary          3.32         2.744347   0.575653
       Romania          3.25         2.521195   0.728805
         Spain          3.14         2.431889   0.708111
Czech Republic          2.61         0.981602   1.628398
        Poland          1.86         1.232609   0.627391

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  Predi